# 07 — Multi-layer hypsometric physics loss (v3)

Joint training v2 showed that the **single-layer** 500–850 hPa hypsometric loss did not help, and that it made the
*unconstrained* lower layers less consistent (700–850 hPa +36 %, 850–925 hPa ×2.4). This notebook tests the fix:

* **multi-layer loss**: the residual-matching hypsometric loss on **every adjacent level pair**
  (925–850, 850–700, 700–600, 600–500, 500–250, 250–50 hPa), so inconsistency cannot be pushed into a neighbouring layer;
* **per-layer scales** (`--physics_scale_m auto`): each layer's loss is divided by the starting model's own residual error
  on a training subset, so every layer counts equally;
* summing the three layers from 850 to 500 hPa gives the 500–850 thickness with a log-pressure trapezoidal mean
  temperature over 500/600/700/850 hPa, so the old layer is still constrained, with a better mean temperature.

| Stage | What | Compared with |
|---|---|---|
| A | adapt the λ=0 encoder to the corrected grid with **weather + multi-layer physics**, 3 seeds | v2 Stage A: weather only (`nophys`) and single-layer physics (`phys`) |
| B | frozen hazard heads on all 9 encoders, logistic baseline, paired 7-day block bootstrap | multi vs nophys and multi vs phys, per seed |

**Attach:** the corrected 6×6 dataset, `checkpoints_for_regional_forecasting_baseline` (λ=0), the IMD rainfall, and
**the output of the `joint_training_v2` notebook** (*Add Input → Notebooks → joint_training_v2*) so its six Stage-A
encoders are reused instead of retrained. Without it, they are retrained here (+~75 min).
**Accelerator:** GPU T4 ×2. Run once with `SMOKE_TEST = True` (~10 min), then set it to `False` and *Save & Run All (Commit)*.
A full run is roughly 1.5 h when the v2 encoders are attached.

In [ ]:
SMOKE_TEST = True
SEEDS = [0, 1, 2]
STAGE_A_EPOCHS = 10
BATCH_SIZE = 128
PHYSICS_LAYERS = "adjacent"        # or e.g. "850-925,700-850,600-700,500-600" (troposphere only)
REUSE_V2_ENCODERS = True           # use joint_training_v2's A_nophys_s* / A_phys_s* encoders if attached
DELETE_FEATURE_CACHE = True

INPUT_ROOT = "/kaggle/input"
CORRECTED_DIR_OVERRIDE = None
IMD_DIR_OVERRIDE = None
NOPHYSICS_CKPT_OVERRIDE = None     # the λ=0 internship checkpoint used to start Stage A
V2_STAGEA_DIR_OVERRIDE = None      # folder containing A_nophys_s0/, A_phys_s0/, ... from joint_training_v2
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
WORK = "/kaggle/working"

In [ ]:
import os, sys, json, subprocess, pathlib, importlib.util, time, shutil
REPO_DIR = f"{WORK}/ClimaX-Monsoon-PINN"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
print("repo commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.6.12"], check=True)
timm_init = pathlib.Path(importlib.util.find_spec("timm").origin).parent / "models" / "__init__.py"
txt = timm_init.read_text()
if sys.version_info >= (3, 11) and "\nfrom .maxxvit import *" in txt:
    timm_init.write_text(txt.replace("\nfrom .maxxvit import *", "\n# from .maxxvit import *  # disabled for py>=3.11"))
sys.path.insert(0, f"{REPO_DIR}/src")
import torch, numpy as np, pandas as pd
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Inputs

In [ ]:
from climax_hazards import inspect_utils as iu
cands = [p for p in iu.find_dataset_dirs(INPUT_ROOT)
         if np.load(p / "lon.npy").size == 6 and not iu.detect_lon_padding(p)["looks_padded"]]
CORR_DIR = str(iu.resolve_single(cands, CORRECTED_DIR_OVERRIDE, "corrected 6x6 dataset"))
def find_imd(root):
    for dirpath, _, files in os.walk(root):
        if "1979.grd" in files:
            return dirpath
IMD_DIR = IMD_DIR_OVERRIDE or find_imd(INPUT_ROOT)
print("corrected dataset:", CORR_DIR, "\nIMD:", IMD_DIR or "NOT FOUND - rainfall skipped")

rows = []
for p in iu.find_checkpoints(INPUT_ROOT):
    if p.name == "best_encoder.ckpt":          # exported Stage-A encoders, handled below
        continue
    ck = iu.load_checkpoint(p)
    mc = next((v for k, v in (ck.get("callbacks") or {}).items() if k.startswith("ModelCheckpoint")), {})
    rows.append({"path": str(p), "run_dir": str(p.parent), "file": p.name, "lambda": ck.get("hyper_parameters", {}).get("physics_lambda"),
                 "epoch": ck.get("epoch"), "best_at_save": os.path.basename(mc.get("best_model_path", "") or "")})
    del ck
ckpts = pd.DataFrame(rows)
final = ckpts.sort_values("epoch").groupby("run_dir").tail(1).set_index("run_dir")
ckpts["run_final_best"] = [r.file == final.loc[r.run_dir, "best_at_save"] for r in ckpts.itertuples()]
if NOPHYSICS_CKPT_OVERRIDE:
    INIT = NOPHYSICS_CKPT_OVERRIDE
else:
    sel = ckpts[(ckpts["lambda"] == 0.0) & ckpts["run_final_best"]]
    assert len(sel) == 1, f"expected one final-best λ=0 checkpoint, found {len(sel)}; set NOPHYSICS_CKPT_OVERRIDE"
    INIT = sel["path"].iloc[0]
print("Stage A starts from (λ=0):", INIT)

def find_v2_stagea(root):
    for dirpath, dirnames, _ in os.walk(root):
        if all(os.path.isfile(f"{dirpath}/A_{v}_s{s}/best_encoder.ckpt") for v in ("nophys", "phys") for s in SEEDS):
            return dirpath
V2_A = (V2_STAGEA_DIR_OVERRIDE or find_v2_stagea(INPUT_ROOT)) if REUSE_V2_ENCODERS else None
print("v2 Stage-A encoders:", V2_A or "not attached - nophys/phys will be retrained here")

In [ ]:
WORK = os.path.abspath(WORK)
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src")
tag = "_smoke" if SMOKE_TEST else ""
seeds = SEEDS[:1] if SMOKE_TEST else SEEDS
def run(cmd, log_path):
    t0 = time.time()
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if not line.startswith(" ") or "seed" in line:
                print(line, end="", flush=True)
        rc = proc.wait()
    print(f"--> exit {rc} after {(time.time() - t0) / 60:.1f} min")
    assert rc == 0, f"failed, see {log_path}"
SUMMARY = f"{WORK}/summary{tag}"
os.makedirs(SUMMARY, exist_ok=True)

## Stage A — multi-layer physics encoders (and v2 encoders, reused or retrained)

In [ ]:
A = {}
variants = ["multi"] + ([] if (V2_A and not SMOKE_TEST) else ["nophys", "phys"])
for variant in variants:
    for seed in seeds:
        name = f"A_{variant}_s{seed}"
        out = f"{WORK}/stageA/{name}{tag}"
        A[name] = out
        if os.path.exists(f"{out}/best_encoder.ckpt"):
            continue
        os.makedirs(out, exist_ok=True)
        cmd = [sys.executable, "scripts/mtech/train_joint.py", "--data_dir", CORR_DIR, "--init_ckpt", INIT, "--out_dir", out,
               "--no_hazards", "--export_encoder", "--balancing", "uncertainty", "--seed", str(seed), "--epochs", str(STAGE_A_EPOCHS),
               "--batch_size", str(BATCH_SIZE), "--patience", "3"]
        if variant == "nophys":
            cmd += ["--no_physics"]
        elif variant == "multi":
            cmd += ["--physics_layers", PHYSICS_LAYERS, "--physics_scale_m", "auto"]
        if SMOKE_TEST:
            cmd += ["--smoke", "--scale_samples", "512"]
        print(f"\n===== {name} =====")
        run(cmd, f"{out}/run.log")
if V2_A and not SMOKE_TEST:
    for variant in ("nophys", "phys"):
        for seed in seeds:
            A[f"A_{variant}_s{seed}"] = f"{V2_A}/A_{variant}_s{seed}"   # read-only input folder
print({k: v for k, v in A.items()})

Test-set weather and hypsometric consistency for all encoders. v2's `weather_test.csv` lacks the new 500–600 hPa layer,
so reused v2 encoders are re-evaluated with `evaluate_forecast.py` (same samples, same metric).

In [ ]:
layers = ["hyps_resid_850_925", "hyps_resid_700_850", "hyps_resid_600_700", "hyps_resid_500_600", "hyps_resid_250_500",
          "hyps_resid_50_250", "hyps_resid_500_850"]
key_vars = ["2m_temperature", "10m_u_component_of_wind", "temperature_850", "geopotential_500", "geopotential_850"] + layers
def weather_csv(name, out):
    f = f"{out}/weather_test.csv"
    if os.path.exists(f) and "hyps_resid_500_600" in set(pd.read_csv(f)["variable"]):
        return f
    f2 = f"{WORK}/reeval/{name}.csv"
    if not os.path.exists(f2):
        os.makedirs(f"{WORK}/reeval", exist_ok=True)
        cmd = [sys.executable, "scripts/mtech/evaluate_forecast.py", "--data_dir", CORR_DIR, "--checkpoint", f"{out}/best_encoder.ckpt", "--out", f2]
        if SMOKE_TEST:
            cmd += ["--max_files", "1"]
        print(f"re-evaluating {name}"); run(cmd, f"{WORK}/reeval/{name}.log")
    return f2

rowsA = []
for name, out in A.items():
    wt = pd.read_csv(weather_csv(name, out))
    if "columns" in wt:
        wt = wt[wt["columns"] == "real"]
    meta = json.load(open(f"{out}/run_meta.json"))
    for _, r in wt.iterrows():
        src = "model" if r["source"] not in ("climatology", "persistence") else r["source"]
        rowsA.append({"run": name, "variant": name.split("_")[1], "seed": int(name.split("_s")[-1]), "source": src,
                      "variable": r["variable"], "rmse": r["rmse"], "selected_epoch": meta["selected_epoch"],
                      "physics_scales_m": json.dumps(meta.get("physics_scales_m"))})
dfA = pd.DataFrame(rowsA)
dfA.to_csv(f"{SUMMARY}/stageA_weather_all.csv", index=False)
m = dfA[(dfA.source == "model") & dfA.variable.isin(key_vars)]
tabA = m.groupby(["variable", "variant"])["rmse"].agg(["mean", "std"]).unstack("variant")
clim = dfA[(dfA.source == "climatology") & dfA.variable.isin(key_vars)].groupby("variable")["rmse"].first()
tabA = pd.concat([tabA, clim.rename(("climatology", ""))], axis=1).reindex(key_vars)
with pd.option_context("display.width", 220, "display.max_columns", 30):
    print("Stage A test RMSE (hyps rows: residual-matching error in m), mean / sd over seeds:")
    print(tabA.round(3).to_string())
tabA.to_csv(f"{SUMMARY}/stageA_weather_table.csv")
means = m.groupby(["variable", "variant"])["rmse"].mean().unstack("variant").reindex(key_vars)
for ref in ("nophys", "phys"):
    if ref in means:
        means[f"multi_vs_{ref}_%"] = 100 * (means["multi"] / means[ref] - 1)
print("\nmulti-layer change vs references (%, negative = better):")
print(means.filter(like="_vs_").round(1).to_string())
means.to_csv(f"{SUMMARY}/stageA_relative_change.csv")
print("\nper-layer physics scales (m) of the multi runs:")
print(dfA[dfA.variant == "multi"].groupby("run")["physics_scales_m"].first().to_string())

## Stage B — frozen hazard heads on all encoders (multi vs nophys, multi vs phys)

In [ ]:
FROZEN_B = f"{WORK}/stageB{tag}"
if not os.path.exists(f"{FROZEN_B}/hazard_summary_test.csv"):
    os.makedirs(FROZEN_B, exist_ok=True)
    cmd = [sys.executable, "scripts/mtech/run_hazard_experiment.py", "--data_dir", CORR_DIR, "--out_dir", FROZEN_B]
    for name, out in A.items():
        cmd += ["--checkpoint", f"{name}={out}/best_encoder.ckpt"]
    for seed in seeds:
        for ref in ("nophys", "phys"):
            if f"A_{ref}_s{seed}" in A:
                cmd += ["--compare", f"A_multi_s{seed}:A_{ref}_s{seed}"]
    if IMD_DIR:
        cmd += ["--imd_dir", IMD_DIR]
    if SMOKE_TEST:
        cmd += ["--max_samples", "300", "--epochs", "2", "--seeds", "0", "--no_logistic"]
    run(cmd, f"{FROZEN_B}/run.log")
    if DELETE_FEATURE_CACHE:
        shutil.rmtree(f"{FROZEN_B}/features", ignore_errors=True)

sB = pd.read_csv(f"{FROZEN_B}/hazard_summary_test.csv")
sB["variant"] = sB["source"].map(lambda s: s.split("_")[1] if s.startswith("A_") else s)
tabB = sB.groupby(["hazard", "variant"])["pr_auc"].agg(["mean", "std", "count"]).round(4)
print("Stage B test PR-AUC (frozen heads), mean (sd) over encoder seeds:")
print(tabB.to_string())
tabB.to_csv(f"{SUMMARY}/stageB_hazard_table.csv")
if os.path.exists(f"{FROZEN_B}/bootstrap_test.csv"):
    bB = pd.read_csv(f"{FROZEN_B}/bootstrap_test.csv")
    print("\nmulti − reference per seed (paired 7-day block bootstrap, 95% CI):")
    print(bB.round(4).to_string(index=False))
    bB.to_csv(f"{SUMMARY}/stageB_bootstrap.csv", index=False)

In [ ]:
# multi-layer encoders vs the logistic baseline (paired bootstrap on the saved test predictions)
from climax_hazards import hazard_model as hm
pdir = f"{FROZEN_B}/preds"
if os.path.exists(f"{pdir}/logistic.npz"):
    t = np.load(f"{pdir}/targets.npz")
    hz_names = json.load(open(f"{FROZEN_B}/run_meta.json"))["hazards"]   # column order of the .npz files
    rows = []
    for seed in seeds:
        a = np.load(f"{pdir}/A_multi_s{seed}.npz")["test"]
        b = np.load(f"{pdir}/logistic.npz")["test"]
        for k, h in enumerate(hz_names):
            rows.append({"a": f"A_multi_s{seed}", "b": "logistic", "hazard": h, **hm.paired_block_bootstrap(a[:, k], b[:, k], t["test_y"][:, k])})
    bl = pd.DataFrame(rows)
    print(bl.round(4).to_string(index=False))
    bl.to_csv(f"{SUMMARY}/stageB_multi_vs_logistic.csv", index=False)
print("\nsummary files:", sorted(os.listdir(SUMMARY)))